# TRIAGE-EG Completion Tonight v1.1 — 38 Final Completion

Exactly B0, M0_v11 and M1_v11. M1 must differ semantically from M0 before GT or the run fails `GRAPH_NOT_EXERCISED`. FULL is evaluated for architectural truth; SAFE preserves B0 Top5 for submission safety.

In [ ]:
import os
from pathlib import Path
REPO_URL="https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git"
REPO_REF="TRIAGEEG"
ANCHOR="d338d8d809bbb9e057e18becc607af2eb3bea254"
REPO_DIR=Path(os.environ.get("AIC_REPO_DIR","/kaggle/working/AIC2026_TeamPTK_SGU"))
RAW_INPUT=Path(os.environ.get("AIC_DATA_ROOT","/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_INPUT=Path(os.environ.get("AIC_TEAM_EVAL_ROOT","/kaggle/input/datasets/irthn1311/aic2026-team-eval-dev-v1"))
FREEZE_INPUT=Path(os.environ.get("AIC_FS1_MASTER_FREEZE_ROOT","/kaggle/input/datasets/irthn1311/fs1-master-preparation-freeze-2026-08-18"))
WHISPER_INPUT=Path(os.environ.get("AIC_WHISPER_ROOT","/kaggle/input/datasets/irthn1311/fs1-whisper-large-v3-turbo-asset"))
QWEN_INPUT=Path(os.environ.get("AIC_QWEN_ROOT","/kaggle/input/datasets/irthn1311/fs1-qwen2-5-vl-3b-instruct-asset"))
EVIDENCE_INPUT=Path(os.environ.get("AIC_COMPLETION_EVIDENCE_ROOT","/kaggle/input/datasets/irthn1311/triage-eg-completion-v11-evidence-bundle"))
OUTPUT_ROOT=Path("/kaggle/working/triage_eg_completion_v11"); OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
OUTPUT_ZIP=Path("/kaggle/working/triage_eg_completion_v11_bundle.zip")
print({"required_inputs":{"raw":str(RAW_INPUT),"team_eval":str(TEAM_INPUT),"freeze":str(FREEZE_INPUT),"evidence_v11":str(EVIDENCE_INPUT),"qwen":str(QWEN_INPUT)},"internet_required":"ONLY_FOR_GIT_CLONE","gpu":"TESLA_T4","output_zip":str(OUTPUT_ZIP),"submissions":["/kaggle/working/submission_FULL.zip","/kaggle/working/submission_SAFE.zip"]})


In [ ]:
import subprocess,sys
if not (REPO_DIR/".git").is_dir():
    subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(["git","fetch","origin",REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(["git","checkout","--detach","FETCH_HEAD"],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
ancestor=subprocess.run(["git","merge-base","--is-ancestor",ANCHOR,HEAD],cwd=REPO_DIR).returncode==0
if not ancestor: raise RuntimeError(f"COMPLETION_V11_LINEAGE_FAIL anchor={ANCHOR} HEAD={HEAD}")
sys.path.insert(0,str(REPO_DIR/"src"))
import torch
GPU={"torch":torch.__version__,"cuda_build":torch.version.cuda,"available":torch.cuda.is_available(),"name":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
if not torch.cuda.is_available() or "T4" not in str(GPU["name"]).upper(): raise RuntimeError(f"COMPLETION_V11_T4_REQUIRED: {GPU}")
print({"HEAD":HEAD,"anchor_is_ancestor":ancestor,"gpu":GPU})


In [ ]:
import json,zipfile
def bounded(root,name,max_depth=6):
    output=[]; root=Path(root)
    if not root.exists(): return output
    for directory,subdirs,files in os.walk(root):
        current=Path(directory); depth=len(current.relative_to(root).parts)
        subdirs[:]=[] if depth>=max_depth else [x for x in subdirs if x not in {".cache","blobs","snapshots"}]
        if name in files: output.append(current/name)
    return output
def unique(values,label):
    values=sorted(set(Path(x).resolve() for x in values))
    if len(values)!=1: raise RuntimeError(f"Expected one {label}; found {values}")
    return values[0]
def find_any(hint,name):
    return unique(bounded(hint,name) or bounded("/kaggle/input",name),name)
def model_root(hint):
    return find_any(hint,"config.json").parent
QWEN_ROOT=model_root(QWEN_INPUT)
TEAM_QUERIES=bounded(TEAM_INPUT,"queries.jsonl") or bounded("/kaggle/input","queries.jsonl")
BENCH={name:unique([p.parent for p in TEAM_QUERIES if p.parent.name==name],name) for name in ("dev_cross_60","dev_l21_150")}
PROTOCOL=find_any(FREEZE_INPUT,"FS1_PROTOCOL.md"); PREP=PROTOCOL.parent
print({"qwen":str(QWEN_ROOT),"benchmarks":{k:str(v) for k,v in BENCH.items()},"freeze":str(PREP)})

EVIDENCE_MANIFEST=find_any(EVIDENCE_INPUT,"evidence_manifest.json"); EVIDENCE_ROOT=EVIDENCE_MANIFEST.parent
PLUGIN_STATUS=json.loads((EVIDENCE_ROOT/"plugin_status.json").read_text())
required={"asr":"PASS","xclip":"PASS","dino":"PASS"}
for name,status in required.items():
    if PLUGIN_STATUS[name]["status"]!=status: raise RuntimeError(f"MANDATORY_PLUGIN_NOT_PASS: {name} {PLUGIN_STATUS[name]}")
if PLUGIN_STATUS["ocr"]["status"] not in {"PASS","OCR_LOCAL_ONLY"}: raise RuntimeError("OCR_NOT_WORKING")


In [ ]:
test_env=os.environ.copy(); test_env["PYTHONPATH"]=str(REPO_DIR/"src")+(os.pathsep+test_env["PYTHONPATH"] if test_env.get("PYTHONPATH") else "")
test=subprocess.run([sys.executable,"-m","pytest","tests/unit/fs1_v11","tests/unit/fs1","tests/unit/bcf1_protected_late_fusion","-q"],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
TEST_SUMMARY={"returncode":test.returncode,"stdout_tail":test.stdout.splitlines()[-20:],"stderr_tail":test.stderr.splitlines()[-20:]}
if test.returncode: raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)


In [ ]:
from triage_eg.fs1.io import PreGTGate,read_jsonl,write_jsonl,sha256
from triage_eg.fs1.runner import group_predictions
from triage_eg.fs1_v11.pipeline import assert_graph_exercised,build_completion_arm,semantic_content_hash
QUERIES={"cross":read_jsonl(BENCH["dev_cross_60"]/"queries.jsonl"),"l21":read_jsonl(BENCH["dev_l21_150"]/"queries.jsonl")}
B0_PATHS={"cross":PREP/"frozen_baseline/cross_b0_bcf1_f1.jsonl","l21":PREP/"frozen_baseline/l21_b0_bcf1_f1.jsonl"}
EXPECTED={"cross":"801e9e4a8e33916cb0430c9c391694410972a84b212d0db949d63671be39e2dc","l21":"3c4dbd2bf4766b286d1efceded120c801e59696d08ab3deb19dd38669074fd16"}
for name,path in B0_PATHS.items():
    if sha256(path)!=EXPECTED[name]: raise RuntimeError(f"B0_REPRODUCTION_FAIL {name}")
B0={name:read_jsonl(path) for name,path in B0_PATHS.items()}


In [ ]:
# Load repaired evidence only; build query/event grouped records.
files={"asr":EVIDENCE_ROOT/"asr_transcripts_v11.jsonl","action":EVIDENCE_ROOT/"xclip_evidence_v11.jsonl","object":EVIDENCE_ROOT/"dino_evidence_v11.jsonl","ocr":EVIDENCE_ROOT/"ocr_records_v11.jsonl"}
EVIDENCE={benchmark:{name:{} for name in (*files,"qwen")} for benchmark in ("cross","l21")}
for modality in ("action","object","ocr"):
    for row in read_jsonl(files[modality]):
        benchmark=row.get("benchmark")
        if benchmark in EVIDENCE: EVIDENCE[benchmark][modality].setdefault(str(row["query_id"]),[]).append(row)
# ASR lexical hits rerank only deterministic B0 identities; no invented IDs.
ASR_INDEX=json.loads((EVIDENCE_ROOT/"asr_lexical_index_v11.json").read_text()); import re
for benchmark in ("cross","l21"):
    grouped=group_predictions(B0[benchmark])
    for query in QUERIES[benchmark]:
        query_id=str(query["query_id"]); tokens=set(re.findall(r"\w+",str(query.get("query","")).casefold())); videos={hit["video_id"] for token in tokens for hit in ASR_INDEX.get(token,[])}; EVIDENCE[benchmark]["asr"][query_id]=[{**row,"source":"asr"} for row in grouped[query_id] if row["video_id"] in videos]


In [ ]:
# Mandatory Qwen QA with answer-type canonicalization and <=100-byte answers.
from PIL import Image
from triage_eg.data.stage0_audit.asset_resolver import discover_layout,resolve_assets
from triage_eg.fs1.qa import GroundingCandidate,bounded_grounding_candidates
from triage_eg.fs1.qwen_adapter import QwenEvidenceAdapter
from triage_eg.fs1_v11.qa import answer_type,canonical_short_answer
from triage_eg.video import OpenCVRawVideoDecoder
video_parts,keyframe_parts=discover_layout(RAW_INPUT); adapter=QwenEvidenceAdapter(QWEN_ROOT); adapter.load(); QA_DIAGNOSTICS=[]
for benchmark in ("cross","l21"):
    grouped=group_predictions(B0[benchmark])
    for query in QUERIES[benchmark]:
        if str(query["task"]).upper()!="QA": continue
        kind=answer_type(str(query["question"])); answers=[]
        for candidate in bounded_grounding_candidates([GroundingCandidate(str(row["video_id"]),int(row["frame_id"]),int(row["rank"]),{"source":"B0"}) for row in grouped[str(query["query_id"])]]) :
            try:
                assets=resolve_assets(RAW_INPUT,candidate.video_id,video_parts,keyframe_parts); decoder=OpenCVRawVideoDecoder(candidate.video_id,assets.video); image=Image.fromarray(decoder.decode_indices([candidate.frame_id])[0].image); decoder.close(); context_rows=EVIDENCE[benchmark]["asr"].get(str(query["query_id"]),[])[:3]+EVIDENCE[benchmark]["ocr"].get(str(query["query_id"]),[])[:3]; evidence_context=" | ".join(str(row.get("text",row.get("normalized_text",""))) for row in context_rows); parsed,audit=adapter.answer(candidate,image,description=str(query.get("query","")),question=str(query["question"]),evidence_context=evidence_context);
                if parsed and parsed["evidence_sufficient"]: parsed["answer"]=canonical_short_answer(parsed["answer"],kind); answers.append({**parsed,"query_id":str(query["query_id"]),"rank":candidate.evidence_rank,"source":"qwen"})
                QA_DIAGNOSTICS.append({"benchmark":benchmark,"query_id":str(query["query_id"]),"answer_type":kind,**audit})
            except Exception as error: QA_DIAGNOSTICS.append({"benchmark":benchmark,"query_id":str(query["query_id"]),"status":"FALLBACK","error":f"{type(error).__name__}: {error}"})
        EVIDENCE[benchmark]["qwen"][str(query["query_id"])]=answers
adapter.unload(); (OUTPUT_ROOT/"qa_diagnostics.jsonl").write_text("".join(json.dumps(row,ensure_ascii=False,default=str)+"\n" for row in QA_DIAGNOSTICS))


In [ ]:
# PRE-GT B0/M0/M1 predictions. Revision provider must return genuine repaired evidence.
import time
gate=PreGTGate(); PRED={}; SAFE={}; DIAGNOSTICS=[]; CONTENT={}; LATENCY={}
for benchmark in ("cross","l21"):
    benchmark_started=time.monotonic()
    b0_path=OUTPUT_ROOT/f"predictions/{benchmark}_B0.jsonl"; b0_path.parent.mkdir(parents=True,exist_ok=True); import shutil; shutil.copy2(B0_PATHS[benchmark],b0_path); gate.finalize(benchmark,"B0",b0_path)
    m0,safe0,diag0=build_completion_arm("M0_v11",QUERIES[benchmark],B0[benchmark],EVIDENCE[benchmark],{"asr","ocr","action","object"})
    def revision_provider(query,event,action):
        query_id=str(query["query_id"]); candidates=[]
        for modality in ("action","object","ocr","asr"):
            candidates.extend(row for row in EVIDENCE[benchmark][modality].get(query_id,[]) if row.get("event_index") in {None,event.event_index})
        unique=[]; seen=set()
        for row in sorted(candidates,key=lambda item:(int(item.get("rank",100)),str(item.get("video_id")),int(item.get("frame_id",item.get("anchor_frame",0))))):
            value={**row,"frame_id":int(row.get("frame_id",row.get("anchor_frame",0))),"source":str(row.get("source","revision"))}; key=(value["video_id"],value["frame_id"],value["source"])
            if key not in seen: seen.add(key); unique.append(value)
        if not unique: raise RuntimeError(f"GRAPH_REVISION_NO_REAL_EVIDENCE {query_id} event={event.event_index}")
        return unique[:10]
    m1,safe1,diag1=build_completion_arm("M1_v11",QUERIES[benchmark],B0[benchmark],EVIDENCE[benchmark],{"asr","ocr","action","object"},revision_provider=revision_provider)
    for arm,rows in (("M0",m0),("M1",m1)):
        path=OUTPUT_ROOT/f"predictions/{benchmark}_{arm}_v11_FULL.jsonl"; write_jsonl(path,rows); gate.finalize(benchmark,arm,path); PRED[(benchmark,arm)]=rows
    for arm,rows in (("M0",safe0),("M1",safe1)):
        safe_path=OUTPUT_ROOT/f"predictions/{benchmark}_{arm}_v11_SAFE.jsonl"; write_jsonl(safe_path,rows); SAFE[(benchmark,arm)]=rows
    DIAGNOSTICS.extend(diag0+diag1); CONTENT[benchmark]={"M0":semantic_content_hash(m0),"M1":semantic_content_hash(m1)}; LATENCY[benchmark]={"prediction_seconds":time.monotonic()-benchmark_started,"query_count":len(QUERIES[benchmark])}
assert_graph_exercised({name:PRED[(name,"M0")] for name in ("cross","l21")},{name:PRED[(name,"M1")] for name in ("cross","l21")},DIAGNOSTICS)
(OUTPUT_ROOT/"prediction_hashes.json").write_text(json.dumps({"bytes":gate.hashes,"semantic_content":CONTENT},indent=2)+"\n")
(OUTPUT_ROOT/"graph_diagnostics.jsonl").write_text("".join(json.dumps(row,ensure_ascii=False,default=str)+"\n" for row in DIAGNOSTICS if row.get("graph")))
(OUTPUT_ROOT/"routing_diagnostics.jsonl").write_text("".join(json.dumps(row,ensure_ascii=False,default=str)+"\n" for row in DIAGNOSTICS))
(OUTPUT_ROOT/"query_event_compilation.jsonl").write_text("".join(json.dumps(event,ensure_ascii=False)+"\n" for row in DIAGNOSTICS for event in row["events"]))


In [ ]:
# GT boundary opens only after graph activation and all six hashes.
gate.open_gt()
from aic2026_eval.io import read_jsonl as eval_read
from aic2026_eval.scoring import evaluate
EVALUATIONS={}
for benchmark,root in (("cross",BENCH["dev_cross_60"]),("l21",BENCH["dev_l21_150"])):
    gt=eval_read(root/"gt.jsonl"); queries=eval_read(root/"queries.jsonl"); EVALUATIONS[benchmark]={}
    for arm,rows in (("B0",B0[benchmark]),("M0_v11",PRED[(benchmark,"M0")]),("M1_v11",PRED[(benchmark,"M1")])):
        summary,per_query,slices,issues=evaluate(queries,rows,gt); EVALUATIONS[benchmark][arm]={"summary":summary,"per_query":per_query,"slices":slices,"issues":issues}
(OUTPUT_ROOT/"official_evaluations.json").write_text(json.dumps(EVALUATIONS,indent=2,default=str)+"\n")
PAIRED_DELTAS={}
for benchmark,arms in EVALUATIONS.items():
    base=arms["B0"]["summary"]; PAIRED_DELTAS[benchmark]={}
    for arm in ("M0_v11","M1_v11"):
        PAIRED_DELTAS[benchmark][arm]={key:arms[arm]["summary"].get(key,0)-base.get(key,0) for key in ("R@1","R@5","R@20","R@50","R@100","final_score") if isinstance(base.get(key), (int,float))}
(OUTPUT_ROOT/"paired_deltas.json").write_text(json.dumps(PAIRED_DELTAS,indent=2)+"\n")


In [ ]:
# Official per-query CSV submissions. M1 is the completed architecture; SAFE keeps B0 Top5.
from triage_eg.submission.aic26_prelim import create_submission_zip
full_rows=PRED[("l21","M1")]; safe_rows=SAFE[("l21","M1")]; full_grouped=group_predictions(full_rows)
all_queries=[{**query,**({"event_count":len(full_grouped[str(query["query_id"])][0]["frame_ids"])} if str(query["task"]).upper()=="TRAKE" and not query.get("event_count") else {})} for query in QUERIES["l21"]]
SUBMISSION_FULL=create_submission_zip(all_queries,full_rows,Path("/kaggle/working/submission_FULL.zip")); SUBMISSION_SAFE=create_submission_zip(all_queries,safe_rows,Path("/kaggle/working/submission_SAFE.zip"))


In [ ]:
ROOT_CAUSE={"asr":"MP4 now decoded by ffmpeg to mono 16k float PCM before Whisper","xclip":"official XCLIPProcessor/XCLIPModel eight-frame logits contract","dino":"official FP32 model and grounded post-processing","ocr":"isolated runtime with working local evidence","event_graph":"exact-N events, per-event evidence, non-noop revision, revised graph state consumed by T3"}
GRAPH_ROWS=[row for row in DIAGNOSTICS if row.get("graph")]; MODALITY_COUNTS={}
for row in DIAGNOSTICS:
    for route in row["routing"]:
        for modality in route["modalities"]: MODALITY_COUNTS[modality]=MODALITY_COUNTS.get(modality,0)+1
COMPLETION_DIAGNOSTICS={"modality_activation_counts":MODALITY_COUNTS,"evidence_records":{name:sum(len(rows) for benchmark in EVIDENCE.values() for rows in benchmark[name].values()) for name in ("asr","ocr","action","object","qwen")},"graph_changed_queries":sum(1 for row in GRAPH_ROWS if row["graph"].get("revision_count")==1),"graph_chain_additions":sum(row["graph"].get("chain_candidates_added",0) for row in GRAPH_ROWS),"graph_missing_event_reduction":sum(len(row["graph"].get("revision",{}).get("missing_before",[]))-len(row["graph"].get("revision",{}).get("missing_after",[])) for row in GRAPH_ROWS),"latency":LATENCY}
RUN_MANIFEST={"HEAD":HEAD,"anchor":ANCHOR,"gt_opened_after_hashes":gate.gt_opened,"graph_exercised":True,"sealed_final_access":False,"post_gt_tuning":False}
for name,value in (("run_manifest.json",RUN_MANIFEST),("plugin_status.json",PLUGIN_STATUS),("evidence_manifest.json",json.loads(EVIDENCE_MANIFEST.read_text())),("tests_summary.json",TEST_SUMMARY),("root_cause_and_fix.json",ROOT_CAUSE),("completion_diagnostics.json",COMPLETION_DIAGNOSTICS)): (OUTPUT_ROOT/name).write_text(json.dumps(value,indent=2,default=str)+"\n")
(OUTPUT_ROOT/"ROOT_CAUSE_AND_FIX_REPORT.md").write_text("# Root Cause and Fix\n\n"+"\n".join(f"- {key}: {value}" for key,value in ROOT_CAUSE.items())+"\n")
(OUTPUT_ROOT/"FORMAL_REPORT.md").write_text("# TRIAGE-EG Completion v1.1\n\nAll mandatory pre-GT hard gates passed. M1 content differs from M0 and graph-revised candidates feed T3. See official_evaluations.json. No post-GT tuning was performed.\n")
import shutil,zipfile
shutil.copy2(SUBMISSION_FULL,OUTPUT_ROOT/"submission_FULL.zip"); shutil.copy2(SUBMISSION_SAFE,OUTPUT_ROOT/"submission_SAFE.zip"); shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")),"zip",OUTPUT_ROOT)
print({"completion_bundle":str(OUTPUT_ZIP),"submission_FULL":str(SUBMISSION_FULL),"submission_SAFE":str(SUBMISSION_SAFE),"evaluations":{b:{a:v["summary"]["final_score"] for a,v in arms.items()} for b,arms in EVALUATIONS.items()}})
